# Get Target

In [2]:
#import necessary dependecies
import os
import warnings
import numpy as np  
import pandas as pd
import random
import gc
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore') 
np.random.seed(111)
random.seed(111)

# Create OOF / Test Pred Files

In [3]:
models_dict = {
    "approach2_swin_small_mixup20": "kaggle-results/approach-2/swin_small_mixup20/",
    "swin_small_newloss_morebands_scheduler_optim_patience20": "kaggle-results/approach-2/swin_small_newloss_morebands_scheduler_optim_patience20/",
    "swin_small_morebands_patience20": "kaggle-results/approach-2/swin_small_morebands_patience20/",
    "approach2_vit_small_patch16_224": "kaggle-results/approach-2/vit_small_patch16_224.augreg_in1k/",
    
}

In [5]:
mapper = {0: "Cocoa", 1: "Palm", 2: "Rubber"}
rev_mapper = {v: k for k, v in mapper.items()}

In [6]:
import numpy as np
import pandas as pd

blend_train_list = []
blend_test_list = []
stack_train_list = []
stack_test_list = []
id_train_all = None  # To keep IDs for stacking
gt_train_all = None  # To keep GT for stacking

class_names = [mapper[i] for i in range(len(mapper))]

for model, result_path in models_dict.items():
    test_oof_path = f"{result_path}/test_oof_proba"
    train_oof_path = f"{result_path}/train_oof_proba"

    # Read 5 folds
    folds = [0,1,2,3,4]
    train_dfs = [pd.read_csv(f"{train_oof_path}/fold{i}.csv") for i in folds]
    test_dfs = [pd.read_csv(f"{test_oof_path}/fold{i}.csv") for i in folds]

    # Convert 'pred_proba' to list of floats
    for df in train_dfs + test_dfs:
        df['probas'] = (df['pred_proba']
                      .str.strip('[]')               # remove [ ]
                      .str.replace(',', ' ')         # unify separators
                      .str.split()                   # split on whitespace
                      .apply(lambda lst: list(map(float, lst))))

    # ------ TRAIN HANDLING ------
    if "approach1" in model:
        # Group by ID and average probas (and get GT for that ID)
        train_concat = pd.concat(train_dfs, axis=0, ignore_index=True)
        avg_train = train_concat.groupby('ID').agg({
            'probas': lambda x: np.mean(np.stack(x), axis=0),
            'gt': 'first'
        }).reset_index()
        train_all = avg_train[['ID', 'gt', 'probas']]
        train_all = train_all.sort_values('ID').reset_index(drop=True)
    else:
        # Default: just concatenate (usual for stacking OOF)
        train_all = pd.concat(train_dfs, axis=0, ignore_index=True)
        train_all = train_all[['ID', 'gt', 'probas']]
        train_all = train_all.sort_values('ID').reset_index(drop=True)

    # For blend: keep the processed DataFrame
    blend_train_list.append(train_all.copy())

    # For stacking: expand probas to columns
    train_probas = pd.DataFrame(train_all['probas'].to_list(), columns=[f"{model}_{c}" for c in class_names])
    stack_train_list.append(train_probas)

    # Save IDs and GT only once (take from the first model processed)
    if id_train_all is None:
        id_train_all = train_all['ID'].reset_index(drop=True)
        gt_train_all = train_all['gt'].reset_index(drop=True)

    # ------ TEST HANDLING (ALWAYS GROUP & AVERAGE) ------
    test_concat = pd.concat(test_dfs, axis=0, ignore_index=True)
    test_avg = test_concat.groupby('ID')['probas'].apply(lambda x: np.mean(np.stack(x), axis=0)).reset_index()
    test_avg = test_avg.sort_values('ID').reset_index(drop=True)

    blend_test_list.append(test_avg.copy())

    test_probas = pd.DataFrame(test_avg['probas'].to_list(), columns=[f"{model}_{c}" for c in class_names])
    stack_test_list.append(test_probas)

# For stacking: concatenate on axis=1, then add ID/gt
stack_train_df = pd.concat([id_train_all, gt_train_all] + stack_train_list, axis=1)
stack_train_df = stack_train_df.rename(columns={0: "ID", 1: "gt"})

# For stacking test: use test_avg's ID
id_test_all = blend_test_list[0]['ID'].reset_index(drop=True)
stack_test_df = pd.concat([id_test_all] + stack_test_list, axis=1)
stack_test_df = stack_test_df.rename(columns={0: "ID"})


In [7]:
print("stack_train_df shape:", stack_train_df.shape)
print("stack_test_df shape:", stack_test_df.shape)

stack_train_df.head()

stack_train_df shape: (953, 14)
stack_test_df shape: (282, 13)


,ID,gt,approach2_swin_small_mixup20_Cocoa,approach2_swin_small_mixup20_Palm,approach2_swin_small_mixup20_Rubber,swin_small_newloss_morebands_scheduler_optim_patience20_Cocoa,swin_small_newloss_morebands_scheduler_optim_patience20_Palm,swin_small_newloss_morebands_scheduler_optim_patience20_Rubber,swin_small_morebands_patience20_Cocoa,swin_small_morebands_patience20_Palm,swin_small_morebands_patience20_Rubber,approach2_vit_small_patch16_224_Cocoa,approach2_vit_small_patch16_224_Palm,approach2_vit_small_patch16_224_Rubber
0,ID_059i1w,1,0.011805,0.715346,0.272849,0.000045,0.915436,0.084519,0.001442,0.897887,0.100671,0.000092,0.873837,0.126071
1,ID_0612VR,1,0.008832,0.968071,0.023097,0.000305,0.962532,0.037163,0.001842,0.959029,0.039129,0.000249,0.999007,0.000745
2,ID_09THeK,2,0.004783,0.071005,0.924212,0.000090,0.000629,0.999280,0.004348,0.051340,0.944313,0.000076,0.001861,0.998063
3,ID_0B09t7,1,0.001971,0.981873,0.016156,0.001246,0.838449,0.160305,0.002675,0.937813,0.059512,0.000106,0.999786,0.000108
4,ID_0DNGt7,0,0.905641,0.071895,0.022464,0.999822,0.000144,0.000035,0.789047,0.103534,0.107419,0.999556,0.000276,0.000168


In [8]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score

# Helper: for a DataFrame with ['ID', 'probas'] column, average probas for each ID
def groupby_id_average_probas(df):
    return df.groupby('ID')['probas'].apply(lambda x: np.mean(np.stack(x), axis=0)).reset_index()

# Process train predictions: group by ID, get average probas per model, and track OOF scores
grouped_blend_train = []
per_model_f1 = []
per_model_acc = []

for i, df in enumerate(blend_train_list):
    avg_df = groupby_id_average_probas(df)
    avg_df = avg_df.merge(df[['ID', 'gt']].drop_duplicates('ID'), on='ID')
    grouped_blend_train.append(avg_df)

    # Get per-model predictions and metrics
    model_pred = np.argmax(np.stack(avg_df['probas'].values), axis=1)
    gt = avg_df['gt'].values
    model_f1 = f1_score(gt, model_pred, average='macro')
    model_acc = accuracy_score(gt, model_pred)
    per_model_f1.append(model_f1)
    per_model_acc.append(model_acc)
    print(f"Model {i+1} | Macro F1: {model_f1:.5f} | Accuracy: {model_acc:.5f}")

# Show average single-model metrics
mean_f1 = np.mean(per_model_f1)
mean_acc = np.mean(per_model_acc)
print(f"\nAverage single-model Macro F1: {mean_f1:.5f}")
print(f"Average single-model Accuracy: {mean_acc:.5f}")

# Compute weights based on OOF F1 (you can use accuracy or other metrics)
weights = np.array(per_model_f1)
weights = weights / np.sum(weights)
print(f"\nWeights based on OOF Macro F1: {weights}")

# Group test set by ID (no ground truth)
grouped_blend_test = []
for df in blend_test_list:
    avg_df = groupby_id_average_probas(df)
    grouped_blend_test.append(avg_df)

# Stack/align by ID, then blend with weights
ids = grouped_blend_train[0]['ID']
gt_labels = grouped_blend_train[0]['gt'].values

# Stack all models' probas for the train set
blend_train_probas = np.stack([
    df.set_index('ID').loc[ids]['probas'].to_list() for df in grouped_blend_train
], axis=0)  # shape: (n_models, n_samples, n_classes)

# Weighted blend
weighted_train_probas = np.tensordot(weights, blend_train_probas, axes=([0], [0]))  # (n_samples, n_classes)
train_pred_labels = np.argmax(weighted_train_probas, axis=1)

f1 = f1_score(gt_labels, train_pred_labels, average='macro')
acc = accuracy_score(gt_labels, train_pred_labels)
print(f"\nWeighted Blended Ensemble Macro F1: {f1:.5f}")
print(f"Weighted Blended Ensemble Accuracy: {acc:.5f}")

# Repeat for test set
test_ids = grouped_blend_test[0]['ID']
blend_test_probas = np.stack([
    df.set_index('ID').loc[test_ids]['probas'].to_list() for df in grouped_blend_test
], axis=0)  # shape: (n_models, n_test_samples, n_classes)
weighted_test_probas = np.tensordot(weights, blend_test_probas, axes=([0], [0]))
test_pred_labels = np.argmax(weighted_test_probas, axis=1)

# Map indices to class names for submission
mapper = {0: "Cocoa", 1: "Palm", 2: "Rubber"}
test_pred_classnames = [mapper[idx] for idx in test_pred_labels]

submission_df = pd.DataFrame({'ID': test_ids, 'Target': test_pred_classnames})
submission_df.to_csv('data/submission/submission_blending.csv', index=False)
print("\nSubmission preview:")
print(submission_df.head())

Model 1 | Macro F1: 0.96246 | Accuracy: 0.96013
Model 2 | Macro F1: 0.96362 | Accuracy: 0.96118
Model 3 | Macro F1: 0.96544 | Accuracy: 0.96327
Model 4 | Macro F1: 0.96501 | Accuracy: 0.96222

Average single-model Macro F1: 0.96413
Average single-model Accuracy: 0.96170

Weights based on OOF Macro F1: [0.24956709 0.24986738 0.25033926 0.25022627]

Weighted Blended Ensemble Macro F1: 0.96700
Weighted Blended Ensemble Accuracy: 0.96432

Submission preview:
          ID  Target
0  ID_000167    Palm
1  ID_004157  Rubber
2  ID_010554  Rubber
3  ID_016218  Rubber
4  ID_018928   Cocoa


In [9]:
print(submission_df["Target"].value_counts())
print(30*"--")
print(submission_df["Target"].value_counts(normalize=True))

Rubber    122
Palm      104
Cocoa      56
Name: Target, dtype: int64
------------------------------------------------------------
Rubber    0.432624
Palm      0.368794
Cocoa     0.198582
Name: Target, dtype: float64


In [11]:
import numpy as np
import pandas as pd

# 1. Collect all model predictions on train set (aligned by ID)
model_preds = []  # Each will be shape (n_samples,)
ids = grouped_blend_train[0]['ID']

for df in grouped_blend_train:
    preds = np.argmax(np.stack(df.set_index('ID').loc[ids]['probas']), axis=1)
    model_preds.append(preds)

# 2. Stack into a matrix: shape (n_models, n_samples)
model_preds_matrix = np.vstack(model_preds)

# 3. Compute correlation matrix between models
correlation_matrix = np.corrcoef(model_preds_matrix)
correlation_df = pd.DataFrame(
    correlation_matrix,
    index=[f'Model_{i+1}' for i in range(model_preds_matrix.shape[0])],
    columns=[f'Model_{i+1}' for i in range(model_preds_matrix.shape[0])]
)

print("\nCorrelation matrix between models' predictions:")
print(correlation_df.round(4))




Correlation matrix between models' predictions:
         Model_1  Model_2  Model_3  Model_4
Model_1   1.0000   0.9702   0.9720   0.9620
Model_2   0.9702   1.0000   0.9687   0.9570
Model_3   0.9720   0.9687   1.0000   0.9671
Model_4   0.9620   0.9570   0.9671   1.0000
